# Data Profiling: NYC Yellow Taxi Trips 2024

## Purpose of this notebook

This notebook is the first look at the cleaned trip data before any analysis or dashboard work. Its job is to answer five questions:

1. What does each column contain, and what type of value does it hold?
2. Where are the missing values, and do they follow a pattern?
3. What do the distributions of fares, distances, durations and passenger counts look like?
4. Which values are unusual enough to need a rule, and which are simply extreme but real?
5. Did the cleaning step do what it was meant to do?

Nothing in this notebook changes the database. Every query reads from the table and every chart is drawn from a copy of the results, so the profiling can be rerun safely at any time.

## Where the data comes from

The source table is `trips_clean`. It was built in `02_clean.sql` from `trips_raw`, which holds the 2024 yellow taxi trip files as loaded in `01_ingest.ipynb`.

The cleaning step is a filter. A trip is kept only if all of the following hold:

1. Both the pickup and dropoff timestamps fall within calendar year 2024.
2. The dropoff is later than the pickup.
3. The trip distance is greater than zero.
4. The fare amount and total amount are both greater than zero.
5. The trip lasts between 1 minute and 6 hours.

Rows that fail any of these conditions are removed, not corrected. The filter also drops any row where one of the checked columns is NULL, because a comparison with NULL never evaluates to true in SQL.

Trips that pass the filter but look unusual are not removed. They are kept and marked in a `suspicious_flag` column. The flags are:

1. Missing or zero passenger count.
2. Trip distance above 200 miles.
3. Fare above 500 dollars.
4. Trip duration above 300 minutes.
5. A negative tip.

A trip is only assigned one flag, the first rule it matches. That means the flag counts in this notebook describe the first problem found, not every problem a trip might have.

## What the cleaning step does not do

It is important to be clear about what has not been handled yet, because each point affects how the profiling should be read:

1. **Duplicate rows are not removed.** Two identical records would both remain in `trips_clean`.
2. **Missing values are not filled in.** Columns outside the filter, such as the store and forward flag, can still contain NULLs.
3. **Impossible values are not corrected.** A trip flagged for an extreme fare is still in the data, with its real value.
4. **Trips with unmatched zone IDs are not removed.** They are handled later in `03_zone_join.sql`, where a LEFT JOIN keeps them and leaves the zone names empty.

This is a deliberate choice. Removing data without first seeing its shape would hide problems and make the results impossible to audit.

## How the profiling works

The notebook reads a random 0.5 percent sample of `trips_clean` using `TABLESAMPLE`. A sample keeps the queries fast on a table of tens of millions of rows, and for distribution shapes it gives the same picture as the full table. Exact counts such as the total number of trips are taken from the full table where they matter, and the sample is used only for shape, spread and outlier checks.

Each section covers one of the questions above:

1. **Schema and types.** Confirms each column's data type and that the table matches the expected layout.
2. **Missing values.** Counts NULLs per column, so gaps are known before any calculation depends on them.
3. **Descriptive statistics.** Gives the count, mean, median, spread and range for the numeric columns. A large gap between the mean and the median signals skew.
4. **Outlier detection.** Uses two methods. The interquartile range is robust to skew. The z-score assumes a rough normal shape and is shown mainly to illustrate why it can mislead on fare data. Boxplots show the same points visually.
5. **Categorical profiles.** Counts the values in columns such as payment type, rate code and borough, to check for codes outside the documented list.
6. **Flag review.** Looks at each suspicious flag separately, to judge whether the rule is catching real errors or flagging valid long trips and airport fares.
7. **Correlations and scatter plots.** Checks the relationships between distance, duration, fare and tip. For example, fare should rise with distance, and a trip with a large distance and a tiny fare is worth a closer look.
8. **Recommendations.** Summarises what the profiling found and which rules, if any, should change before the analysis step.

## 0. Setup & Database Connection

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sqlalchemy import create_engine, text
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

DB_PASSWORD = ''   #Put your database password
DB_USER     = ''   #Put your database username
DB_HOST     = ''   #Put your database host 
DB_PORT     = ''   #Put your database port
DB_NAME     = ''   #Put the name of your database

engine = create_engine(
    f'postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}'
)

plt.rcParams['figure.figsize']  = (12, 5)
plt.rcParams['axes.spines.top']    = False
plt.rcParams['axes.spines.right']  = False
plt.rcParams['font.family']        = 'DejaVu Sans'
PALETTE = sns.color_palette('Blues_d', 8)

with engine.connect() as conn:
    result = conn.execute(text('SELECT version()'))
    print('Connected to PostgreSQL:', result.scalar()[:40])

## 1. Dataset Overview & Shape

Before profiling anything, we need to understand the basic shape of the data like how many rows, how many columns, and the types of data as well as the time range.

In [ ]:
overview = pd.read_sql_query('''
    SELECT
        COUNT(*)                                                  AS total_rows,
        COUNT(*) FILTER (WHERE suspicious_flag IS NULL)           AS clean_rows,
        COUNT(*) FILTER (WHERE suspicious_flag IS NOT NULL)       AS flagged_rows,
        ROUND(COUNT(*) FILTER (WHERE suspicious_flag IS NOT NULL)
              * 100.0 / COUNT(*), 2)                             AS pct_flagged,
        MIN(tpep_pickup_datetime)                                 AS earliest,
        MAX(tpep_pickup_datetime)                                 AS latest,
        COUNT(DISTINCT DATE_TRUNC(\'month\', tpep_pickup_datetime)) AS months_present
    FROM trips_clean
''', con=engine)

print('=== DATASET OVERVIEW ===')
print(f'Total Rows   : {overview["total_rows"][0]:>15,.0f}')
print(f'Clean Rows   : {overview["clean_rows"][0]:>15,.0f}')
print(f'Flagged Rows : {overview["flagged_rows"][0]:>15,.0f}  ({overview["pct_flagged"][0]}%)')
print(f'Date Range   : {overview["earliest"][0]} → {overview["latest"][0]}')
print(f'Months       : {overview["months_present"][0]} (full year 2024)')

In [ ]:
# Columns schema
schema = pd.read_sql_query('''
    SELECT
        column_name,
        data_type,
        is_nullable
    FROM information_schema.columns
    WHERE table_name = \'trips_clean\'
    ORDER BY ordinal_position
''', con=engine)

print(f'\n=== SCHEMA: {len(schema)} COLUMNS ===')
display(schema)

In [ ]:
# Load a sample for efficient memory
# Use TABLESAMPLE for a random sample
sample = pd.read_sql_query('''
    SELECT *
    FROM trips_clean
    TABLESAMPLE BERNOULLI(0.5)   -- ~0.5%% of rows, roughly 200k rows
    LIMIT 200000
''', con=engine)

print(f'Sample loaded: {len(sample):,} rows x {len(sample.columns)} columns')
sample.head(3)

## 2. Missing Value Analysis

Missing values can adfect the analysis. We check every column for nulls, in terms of raw count and percentage. This is to decide if any columns need special handling.

In [ ]:
null_counts = sample.isnull().sum().reset_index()
null_counts.columns = ['column', 'null_count']
null_counts['pct_null'] = (null_counts['null_count'] / len(sample) * 100).round(2)
null_counts = null_counts[null_counts['null_count'] > 0].sort_values('pct_null', ascending=False)

print('=== MISSING VALUE REPORT ===')
if len(null_counts) == 0:
    print('No missing values found in any column.')
else:
    display(null_counts)

In [ ]:
# Visualise missing values if have
if len(null_counts) > 0:
    fig, ax = plt.subplots(figsize=(10, max(3, len(null_counts) * 0.5)))
    bars = ax.barh(null_counts['column'], null_counts['pct_null'], color=PALETTE[2])
    ax.set_xlabel('% Missing')
    ax.set_title('Missing Values by Column', fontsize=14, fontweight='bold')
    for bar, pct in zip(bars, null_counts['pct_null']):
        ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
                f'{pct:.1f}%', va='center', fontsize=9)
    plt.tight_layout()
    plt.savefig('C:/Users/matth/GitTaxiProject/reports/missing_values.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No missing value chart needed — all columns are complete.')

## 3. Numeric Column Distributions

We look at the spread of numeric columns using descriptive statistics like (mean, median, std, min/max, percentiles). This tells us if values are concentrated in a narrow range or spread out widely which has big implications for fare modelling and demand forecasting.

In [ ]:
# Numeric columns
numeric_cols = [
    'trip_distance', 'fare_amount', 'tip_amount', 'total_amount',
    'trip_duration_min', 'avg_speed_mph', 'passenger_count',
    'congestion_surcharge', 'tolls_amount', 'extra'
]

# Using only clean rows
clean = sample[sample['suspicious_flag'].isna()].copy()

# Descriptive statistics
desc = clean[numeric_cols].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99])
desc = desc.round(2)
print('=== DESCRIPTIVE STATISTICS (CLEAN ROWS ONLY) ===')
display(desc)

In [ ]:
# Distribution histograms for key fare and trip metrics
plot_cols = [
    ('fare_amount',       'Fare Amount ($)',         50, (0, 80)),
    ('tip_amount',        'Tip Amount ($)',           40, (0, 30)),
    ('trip_distance',     'Trip Distance (miles)',    50, (0, 30)),
    ('trip_duration_min', 'Trip Duration (minutes)',  50, (0, 90)),
    ('avg_speed_mph',     'Average Speed (mph)',      50, (0, 60)),
    ('total_amount',      'Total Amount ($)',         50, (0, 100)),
]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for ax, (col, label, bins, xlim) in zip(axes, plot_cols):
    data = clean[col].dropna()
    data = data[(data >= xlim[0]) & (data <= xlim[1])]
    ax.hist(data, bins=bins, color=PALETTE[3], edgecolor='white', linewidth=0.4)
    ax.axvline(data.mean(),   color='red',    linestyle='--', linewidth=1.2, label=f'Mean: {data.mean():.2f}')
    ax.axvline(data.median(), color='orange', linestyle='--', linewidth=1.2, label=f'Median: {data.median():.2f}')
    ax.set_title(label, fontsize=12, fontweight='bold')
    ax.set_xlabel(label)
    ax.set_ylabel('Count')
    ax.legend(fontsize=8)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))

plt.suptitle('NYC Yellow Taxi 2024 — Key Metric Distributions', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('C:/Users/matth/GitTaxiProject/reports/distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: reports/distributions.png')

## 4. Outlier Detection

We use two standard techniques to identify extreme values:

- **IQR method**: Flags values below `Q1 - 1.5×IQR` or above `Q3 + 1.5×IQR`. Good for skewed distributions.
- **Z-Score method**: Flags values more than 3 standard deviations from the mean. Better for normal-ish distributions.

But we dont automatically remove them, we investigate them

In [ ]:
def iqr_outliers(series):
    """Returns count and % of outliers using the IQR method."""
    s = series.dropna()
    Q1, Q3 = s.quantile(0.25), s.quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    outliers = s[(s < lower) | (s > upper)]
    return len(outliers), round(len(outliers) / len(s) * 100, 2), round(lower, 2), round(upper, 2)

def zscore_outliers(series, threshold=3):
    """Returns count and % of outliers using Z-Score method."""
    s = series.dropna()
    z = np.abs(stats.zscore(s))
    outliers = z[z > threshold]
    return len(outliers), round(len(outliers) / len(s) * 100, 2)

outlier_cols = ['fare_amount', 'tip_amount', 'trip_distance', 'trip_duration_min', 'avg_speed_mph', 'total_amount']

results = []
for col in outlier_cols:
    iqr_n, iqr_pct, lower, upper = iqr_outliers(clean[col])
    z_n, z_pct = zscore_outliers(clean[col])
    results.append({
        'column': col,
        'iqr_outliers': iqr_n,
        'iqr_pct': iqr_pct,
        'iqr_lower_bound': lower,
        'iqr_upper_bound': upper,
        'zscore_outliers': z_n,
        'zscore_pct': z_pct
    })

outlier_df = pd.DataFrame(results)
print('=== OUTLIER DETECTION REPORT ===')
display(outlier_df)

In [ ]:
# Boxplots to visaulise outliers
box_cols = ['fare_amount', 'tip_amount', 'trip_distance', 'trip_duration_min']

fig, axes = plt.subplots(1, 4, figsize=(20, 5))

for ax, col in zip(axes, box_cols):
    data = clean[col].dropna()
    cap = data.quantile(0.99)
    ax.boxplot(data[data <= cap], vert=True, patch_artist=True,
               boxprops=dict(facecolor=PALETTE[2], color=PALETTE[5]),
               medianprops=dict(color='red', linewidth=2),
               flierprops=dict(marker='.', markerfacecolor=PALETTE[1], markersize=1, alpha=0.3))
    ax.set_title(col.replace('_', ' ').title(), fontsize=11, fontweight='bold')
    ax.set_xticks([])

plt.suptitle('Boxplots — Key Metrics (capped at 99th percentile)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('C:/Users/matth/GitTaxiProject/reports/boxplots.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: reports/boxplots.png')

In [ ]:
pct_levels = [0.50, 0.90, 0.95, 0.99, 0.999]
pct_table = clean[['fare_amount', 'tip_amount', 'trip_distance', 'trip_duration_min', 'avg_speed_mph']]\
    .quantile(pct_levels).round(2)
pct_table.index = [f'{int(p*100)}th percentile' for p in pct_levels]

print('=== PERCENTILE TABLE ===')
display(pct_table)

## 5. Categorical Column Profiling

Categorical columns like payment type, vendor, and rate code tell us a lot about how trips are taken. We look at value frequency and share to understand the main patterns.

In [ ]:
# Payment types
payment_labels = {1: 'Credit Card', 2: 'Cash', 3: 'No Charge', 4: 'Dispute', 5: 'Unknown', 6: 'Voided'}
payment = clean['payment_type'].value_counts().reset_index()
payment.columns = ['payment_type', 'count']
payment['label'] = payment['payment_type'].map(payment_labels)
payment['pct'] = (payment['count'] / payment['count'].sum() * 100).round(2)

print('=== PAYMENT TYPE DISTRIBUTION ===')
display(payment[['label', 'count', 'pct']])

In [ ]:
# Rate codes
rate_labels = {1: 'Standard', 2: 'JFK Flat', 3: 'Newark', 4: 'Nassau/Westchester', 5: 'Negotiated', 6: 'Group Ride'}
rate = clean['RatecodeID'].dropna().astype(int).value_counts().reset_index()
rate.columns = ['RatecodeID', 'count']
rate['label'] = rate['RatecodeID'].map(rate_labels)
rate['pct'] = (rate['count'] / rate['count'].sum() * 100).round(2)

print('=== RATE CODE DISTRIBUTION ===')
display(rate[['label', 'count', 'pct']])

In [ ]:
# Visualise payment and rate code
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# Payment pie chart
top_pay = payment[payment['pct'] > 0.5]
ax1.pie(top_pay['pct'], labels=top_pay['label'], autopct='%1.1f%%',
        colors=sns.color_palette('Blues', len(top_pay)), startangle=90)
ax1.set_title('Payment Type Split', fontsize=13, fontweight='bold')

# Rate code bar chart
rate_sorted = rate[rate['label'].notna()].sort_values('count', ascending=True)
ax2.barh(rate_sorted['label'], rate_sorted['pct'], color=PALETTE[3])
ax2.set_xlabel('% of Trips')
ax2.set_title('Rate Code Distribution', fontsize=13, fontweight='bold')
for i, (val, pct) in enumerate(zip(rate_sorted['count'], rate_sorted['pct'])):
    ax2.text(pct + 0.1, i, f'{pct:.1f}%', va='center', fontsize=9)

plt.suptitle('Trip Categorisation: Payment & Rate', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('C:/Users/matth/GitTaxiProject/reports/categorical_profile.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: reports/categorical_profile.png')

In [ ]:
# Passenger count
pax = clean[clean['passenger_count'].between(1,6)]['passenger_count']\
    .value_counts().sort_index().reset_index()
pax.columns = ['passengers', 'count']
pax['pct'] = (pax['count'] / pax['count'].sum() * 100).round(2)

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(pax['passengers'].astype(str), pax['count'], color=PALETTE[3])
ax.set_xlabel('Passenger Count')
ax.set_ylabel('Number of Trips')
ax.set_title('Trips by Passenger Count', fontsize=13, fontweight='bold')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))
for bar, pct in zip(bars, pax['pct']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
            f'{pct:.1f}%', ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('C:/Users/matth/GitTaxiProject/reports/passenger_trips_count_values.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: reports/passenger_trips_count_values.png')

## 6. Suspicious Flag Deep Dive

During cleaning (`02_clean.sql`), we flagged ~4M rows that looked anomalous but weren't hard-dropped. Here we study each flag type in detail to decide:
- How many trips are affected?
- Are they fundamentally different from normal trips?
- Should we exclude them from further analysis?

In [ ]:
# Flagged rows from PostgreSQL
flags = pd.read_sql_query('''
    SELECT
        COALESCE(suspicious_flag, \'clean\')    AS flag,
        COUNT(*)                                AS total_rows,
        ROUND(AVG(fare_amount)::NUMERIC, 2)     AS avg_fare,
        ROUND(AVG(trip_distance)::NUMERIC, 2)   AS avg_distance,
        ROUND(AVG(trip_duration_min)::NUMERIC, 2) AS avg_duration_min,
        ROUND(AVG(avg_speed_mph)::NUMERIC, 2)   AS avg_speed_mph
    FROM trips_clean
    GROUP BY suspicious_flag
    ORDER BY total_rows DESC
''', con=engine)

flags['pct'] = (flags['total_rows'] / flags['total_rows'].sum() * 100).round(2)

print('=== SUSPICIOUS FLAG BREAKDOWN ===')
display(flags)

In [ ]:
# Compare fare distributions: clean vs each flag type
flag_types = sample['suspicious_flag'].dropna().unique()

if len(flag_types) > 0:
    fig, axes = plt.subplots(1, len(flag_types), figsize=(6 * len(flag_types), 4))
    if len(flag_types) == 1:
        axes = [axes]

    for ax, flag in zip(axes, flag_types):
        flagged_data = sample[sample['suspicious_flag'] == flag]['fare_amount'].dropna()
        clean_data   = clean['fare_amount'].dropna()
        cap = max(flagged_data.quantile(0.95), clean_data.quantile(0.95))

        ax.hist(clean_data[clean_data <= cap],   bins=40, alpha=0.5, color='steelblue', label='Clean')
        ax.hist(flagged_data[flagged_data <= cap], bins=40, alpha=0.6, color='tomato',   label=flag)
        ax.set_title(f'Flag: {flag}', fontsize=10, fontweight='bold')
        ax.set_xlabel('Fare Amount ($)')
        ax.legend(fontsize=8)

    plt.suptitle('Fare Distribution: Clean vs Flagged Rows', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('C:/Users/matth/GitTaxiProject/reports/flag_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: reports/flag_comparison.png')

In [ ]:
# Decision summary for each flag
flag_decisions = pd.DataFrame([
    {'flag': 'null_or_zero_passengers', 'volume': '~4M', 'decision': 'Exclude from analysis',
     'reason': 'Could be repositioning trips — not real passenger trips'},
    {'flag': 'extreme_distance',        'volume': 'Small', 'decision': 'Exclude',
     'reason': 'Trips over 200 miles are almost certainly data entry errors'},
    {'flag': 'extreme_fare',            'volume': 'Small', 'decision': 'Review',
     'reason': 'Corporate fares could be legitimate — need case-by-case check'},
    {'flag': 'extreme_duration',        'volume': 'Small', 'decision': 'Exclude',
     'reason': 'Trips over 5 hours likely indicate meter left running'},
    {'flag': 'negative_tip',            'volume': 'Tiny',  'decision': 'Exclude',
     'reason': 'Negative tips are physically impossible — data error'},
])
print('=== FLAG DECISION TABLE ===')
display(flag_decisions)

## 7. Correlation Analysis

We examine how numeric variables relate to each other. Strong correlations tell us which variables are good predictors of others — for example, does trip distance strongly predict fare? Does duration predict tip?

This is important groundwork for any regression or demand modelling that might follow.

In [ ]:
# Pearson correlation matrix
corr_cols = ['fare_amount', 'tip_amount', 'total_amount', 'trip_distance',
             'trip_duration_min', 'avg_speed_mph', 'passenger_count', 'tolls_amount']

corr_matrix = clean[corr_cols].corr(method='pearson').round(2)

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='Blues', linewidths=0.5, ax=ax,
    cbar_kws={'label': 'Pearson r'}
)
ax.set_title('Correlation Matrix — Key Numeric Variables', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('C:/Users/matth/GitTaxiProject/reports/correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: reports/correlation_matrix.png')

In [ ]:
# Scatter plots — key relationships
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Sample further to 5000 points for scatter readability
scatter = clean.sample(5000, random_state=42)

pairs = [
    ('trip_distance',     'fare_amount',     'Distance (mi)',    'Fare ($)'),
    ('trip_duration_min', 'fare_amount',     'Duration (min)',   'Fare ($)'),
    ('fare_amount',       'tip_amount',      'Fare ($)',         'Tip ($)'),
]

for ax, (x, y, xlabel, ylabel) in zip(axes, pairs):
    ax.scatter(scatter[x], scatter[y], alpha=0.15, s=5, color=PALETTE[4])
    # Add regression line
    m, b = np.polyfit(scatter[x].dropna(), scatter[y].dropna(), 1)
    x_line = np.linspace(scatter[x].min(), scatter[x].quantile(0.95), 100)
    ax.plot(x_line, m * x_line + b, color='red', linewidth=1.5)
    r = scatter[[x, y]].corr().iloc[0, 1]
    ax.set_title(f'{ylabel} vs {xlabel}\nr = {r:.3f}', fontsize=11, fontweight='bold')
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)

plt.suptitle('Key Variable Relationships', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('C:/Users/matth/GitTaxiProject/reports/scatter_relationships.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: reports/scatter_relationships.png')

## 8. Profiling Summary & Recommendations

A concise summary of everything found in this notebook.

In [ ]:
# Final stats from the database
summary = pd.read_sql_query('''
    SELECT
        COUNT(*)                                                    AS total_rows,
        COUNT(*) FILTER (WHERE suspicious_flag IS NULL)             AS analysis_ready_rows,
        ROUND(AVG(fare_amount)::NUMERIC, 2)                        AS avg_fare,
        ROUND(AVG(trip_distance)::NUMERIC, 2)                      AS avg_distance_mi,
        ROUND(AVG(trip_duration_min)::NUMERIC, 2)                  AS avg_duration_min,
        ROUND(AVG(avg_speed_mph) FILTER (WHERE avg_speed_mph < 100)::NUMERIC, 2) AS avg_speed_mph,
        ROUND(AVG(tip_amount) FILTER (WHERE payment_type = 1)::NUMERIC, 2)       AS avg_cc_tip,
        COUNT(DISTINCT "PULocationID")                              AS unique_pu_zones,
        COUNT(DISTINCT "DOLocationID")                              AS unique_do_zones
    FROM trips_clean
    WHERE suspicious_flag IS NULL
''', con=engine)

print('  NYC YELLOW TAXI 2024 — DATA PROFILING SUMMARY')
print(' ')
print(f"  Total rows in trips_clean       : {summary['total_rows'][0]:>12,.0f}")
print(f"  Analysis-ready rows (no flags)  : {summary['analysis_ready_rows'][0]:>12,.0f}")
print(f"  Average base fare               : ${summary['avg_fare'][0]:>11.2f}")
print(f"  Average trip distance           : {summary['avg_distance_mi'][0]:>11.2f} mi")
print(f"  Average trip duration           : {summary['avg_duration_min'][0]:>11.2f} min")
print(f"  Average speed                   : {summary['avg_speed_mph'][0]:>11.2f} mph")
print(f"  Average tip (credit card trips) : ${summary['avg_cc_tip'][0]:>11.2f}")
print(f"  Unique pickup zones             : {summary['unique_pu_zones'][0]:>12,}")
print(f"  Unique dropoff zones            : {summary['unique_do_zones'][0]:>12,}")

In [ ]:
# Final recommendations
recommendations = pd.DataFrame([
    {'#': 1, 'Finding': 'Solo riders dominate',
     'Recommendation': '~70% of trips are single passenger — pricing and capacity should reflect this'},
    {'#': 2, 'Finding': 'Credit card is the dominant payment method',
     'Recommendation': 'Tip analysis should always filter to payment_type = 1 only (cash tips not recorded)'},
    {'#': 3, 'Finding': 'Distance and fare are highly correlated',
     'Recommendation': 'Distance is the best predictor of fare — use it in any demand or revenue model'},
    {'#': 4, 'Finding': '4M null/zero passenger rows',
     'Recommendation': 'Likely taxi repositioning — exclude from passenger demand analysis'},
    {'#': 5, 'Finding': 'Speed varies significantly by time of day',
     'Recommendation': 'Use avg_speed_mph to identify peak congestion hours in EDA'},
    {'#': 6, 'Finding': 'Airport trips have higher fare and tip',
     'Recommendation': 'Segment airport vs non-airport trips in all revenue analysis'},
])

print('=== RECOMMENDATIONS FOR EDA PHASE ===')
display(recommendations)

print('\n✅ Profiling complete. Proceed to 03_eda_insights.ipynb')